# PoC — climate.gg 8단계 옥상 파이프라인

**목적**: HAR 분석으로 발견한 5개 API + WFS 가 라이브 환경에서도 동작하는지 화성/수원 옥상 1동에 대해 끝까지 재현.

**입력**: WGS84 좌표 1개 (`apps/web` 의 VWorld 지도가 4326을 쓰므로 사용자/개발자 인터페이스 모두 4326 으로 통일).

**API 호출 직전에만 EPSG:5186 으로 변환**, 응답을 받아 다시 4326 으로 변환해 저장 → 풀스택은 4326만 받는다.

**산출**: `data/processed/poc/{unq_id}/bundle.json` + `panels_4326.geojson`

**상위 명세**: `docs/api_spec_climate_har.md`

In [ ]:
import json, os, statistics, requests
from pyproj import Transformer
from shapely.geometry import shape, box

# ---- 설정 ----
CLICK_WGS = (127.02948714, 37.26726989)   # (lon, lat) - HAR 검증 좌표 (수원 권선구 인계동)
CELL_W, CELL_H = 1.0, 3.5                  # climate.gg 셀 (m, EPSG:5186)
PANEL_CAPACITY_W = 640
PANEL_ANGLE = '35'
PANEL_TYPE = 1
CELLS_PER_PANEL = 2                        # 1패널 ≈ 셀 2장 (잠정 가정)
OUT_BASE = '../data/processed/poc'

BASE = 'https://climate.gg.go.kr'
HEADERS = {
    'Accept': 'application/json, text/javascript, */*; q=0.01',
    'X-Requested-With': 'XMLHttpRequest',
    'Origin': BASE, 'Referer': BASE + '/',
    'User-Agent': 'solarmate-poc/0.1',
}
FORM_HDR = {**HEADERS, 'Content-Type': 'application/x-www-form-urlencoded; charset=UTF-8'}
JSON_HDR = {**HEADERS, 'Content-Type': 'application/json; charset=UTF-8'}

to_5186 = Transformer.from_crs('EPSG:4326', 'EPSG:5186', always_xy=True)
to_4326 = Transformer.from_crs('EPSG:5186', 'EPSG:4326', always_xy=True)

lon, lat = CLICK_WGS
x5186, y5186 = to_5186.transform(lon, lat)
print(f'WGS84({lon:.6f},{lat:.6f}) -> EPSG:5186({x5186:.3f},{y5186:.3f})')

## Step 1 — selectBuld: 좌표 → 옥상 polygon (EPSG:5186)

In [ ]:
r = requests.post(f'{BASE}/gcs/book/cmm/selectBuld.do',
                  data={'x': f'{x5186}', 'y': f'{y5186}', 'type': 'PANEL'},
                  headers=FORM_HDR, timeout=15)
raw = json.loads(r.text)
assert raw.get('buld'), '건물 미적중'
feature = json.loads(raw['buld']['feature'])
roof_5186 = shape(feature['geometry'])
print('status:', r.status_code, 'vertices:', len(feature['geometry']['coordinates'][0]),
      'area:', round(roof_5186.area, 1), 'sqm')
roof_5186.bounds

## Step 2 — WFS TM_BLDG_INFO: 좌표 → 건물 메타 (unq_id, 높이, 면적, 용도)

In [ ]:
wfs_url = (f'{BASE}/geoserver/spggcee/ows?service=WFS&version=1.0.0&request=GetFeature'
           f'&typeName=spggcee:TM_BLDG_INFO&outputFormat=application/json&SRS=EPSG:5186'
           f'&CQL_FILTER=INTERSECTS(shape,%20Point({x5186}%20{y5186}))')
wfs = requests.get(wfs_url, headers=HEADERS, timeout=15).json()
props = wfs['features'][0]['properties']
unq_id = props['unq_id']
{k: props.get(k) for k in ['unq_id','bldg_nm','bldg_hgt','bdar','bldg_nofl','use_aprv_ymd','bldg_usg_cd','sigun_cd']}

## Step 3 — 옥상 polygon → 1m × 3.5m 셀 격자

climate.gg 가 음영을 계산하는 단위. centroid-in-polygon 으로 옥상 내부만 채택.

In [ ]:
minx, miny, maxx, maxy = roof_5186.bounds
cells = []
pid = 0
y = miny
while y + CELL_H <= maxy:
    x = minx
    while x + CELL_W <= maxx:
        cell = box(x, y, x + CELL_W, y + CELL_H)
        if roof_5186.contains(cell.centroid):
            cells.append((pid, x, y, x + CELL_W, y + CELL_H))
        pid += 1
        x += CELL_W
    y += CELL_H
print('옥상 내부 셀:', len(cells))

## Step 4 — selectSunList: 셀별 음영점수 ⭐

climate.gg 음영 모델 그 자체. 응답 형식 `["{id}|{score}", ...]`.

In [ ]:
panel_params = [('panel', f'{pid}-{x1},{y1},{x2},{y2}') for pid,x1,y1,x2,y2 in cells]
panel_params.append(('type', 'build'))
r = requests.post(f'{BASE}/gcs/panel/selectSunList.do',
                  data=panel_params, headers=FORM_HDR, timeout=30)
sun_raw = json.loads(r.text)
shading = {int(item.split('|')[0]): float(item.split('|')[1]) for item in sun_raw}
vals = list(shading.values())
shading_avg = statistics.mean(vals)
print(f'반환 {len(shading)}/{len(cells)}, min={min(vals):.3f}, mean={shading_avg:.3f}, max={max(vals):.3f}')

## Step 5 — selectBuldInfo: unq_id → 월별 사용량 (10개월)

In [ ]:
r = requests.post(f'{BASE}/gcs/panel/selectBuldInfo.do',
                  data={'unq_id': unq_id}, headers=FORM_HDR, timeout=15)
info = json.loads(r.text)['list'][0]
el = [int(v) for v in (info.get('elpwr_usqty') or '').split(',') if v]
gas = [int(v) for v in (info.get('gas_usqty') or '').split(',') if v]
labels = (info.get('use_ym') or '').replace('\n','-').split(',')
print('전력:', el)
print('가스:', gas)

## Step 6 — selectRuleList: 옥상 polygon → 19종 규제 매칭

In [ ]:
ring = roof_5186.exterior.coords
wkt = 'MULTIPOLYGON(((' + ', '.join(f'{x} {y}' for x,y in ring) + ')))'
r = requests.post(f'{BASE}/gcs/panel/selectRuleList.do',
                  data={'text': wkt}, headers=FORM_HDR, timeout=15)
rules = json.loads(r.text)['list']
hits_pos = [(item['layer'], item['cnt']) for item in rules if item['cnt'] > 0]
print(f'양성 매칭: {hits_pos or "없음"}')

## Step 7 — pv/analysis: 발전·경제성 시뮬

셀 음영 평균과 패널 개수를 입력으로 climate.gg 표준 시뮬 호출.

In [ ]:
panel_count = max(1, len(cells) // CELLS_PER_PANEL)
payload = {
    'latitude': lat, 'longitude': lon,
    'shading_index_average': shading_avg,
    'solar_panel_angle': PANEL_ANGLE,
    'solar_panel_info': {
        'panel_capacity': PANEL_CAPACITY_W,
        'panel_count': panel_count,
        'panel_type': PANEL_TYPE,
    },
}
r = requests.post(f'{BASE}/spsvc/pv/analysis', json=payload, headers=JSON_HDR, timeout=15)
pv = json.loads(r.text)['data']
{
    'install_kw': pv['expected_revenue']['install_kw'],
    'annual_generation_kwh': pv['annual_generation'],
    'first_year_revenue_krw': pv['expected_revenue']['first_year_revenue'],
    'first_year_save_cost_krw': pv['expected_revenue']['first_year_save_cost'],
    'expected_investment_krw': pv['expected_revenue']['expected_investment'],
    'monthly_kwh': [round(m['generation']) for m in pv.get('monthly_generation', [])],
}

## Step 8 — 결과를 EPSG:4326 으로 변환 + 저장

**개발자 인터페이스 규약**: API 응답의 5186 좌표는 분석가가 모두 4326 으로 변환해 저장. 풀스택은 4326만 받는다.

In [ ]:
safe_id = (unq_id or 'unknown').replace('/', '_')
out = f'{OUT_BASE}/{safe_id}'
os.makedirs(out, exist_ok=True)

roof_4326_coords = [list(to_4326.transform(x, y)) for x, y in roof_5186.exterior.coords]

panels_features = []
for pid, x1, y1, x2, y2 in cells:
    if pid not in shading: continue
    ring = [(x1,y1),(x2,y1),(x2,y2),(x1,y2),(x1,y1)]
    ring_4326 = [list(to_4326.transform(px, py)) for px, py in ring]
    panels_features.append({
        'type': 'Feature',
        'geometry': {'type':'Polygon','coordinates':[ring_4326]},
        'properties': {'cell_id': pid, 'shading_score': shading[pid], 'cell_5186_bbox': [x1,y1,x2,y2]},
    })

bundle = {
    'meta': {**{k:props.get(k) for k in ['unq_id','bldg_nm','bldg_hgt','bdar','bldg_nofl','use_aprv_ymd','bldg_usg_cd','sigun_cd']},
             'click_wgs84': {'longitude': lon, 'latitude': lat}},
    'roof_polygon_4326': {'type':'Polygon','coordinates':[roof_4326_coords]},
    'roof_area_sqm_5186': round(roof_5186.area, 2),
    'shading': {'cell_w_m': CELL_W, 'cell_h_m': CELL_H,
                'cells_total': len(cells), 'cells_with_score': len(shading),
                'score_min': min(vals), 'score_mean': shading_avg, 'score_max': max(vals)},
    'usage_monthly': {'labels': labels, 'electricity_kwh': el, 'gas_m3': gas},
    'regulation_hits': hits_pos,
    'pv_analysis_input': payload,
    'pv_analysis_output': pv,
}
with open(f'{out}/bundle.json','w',encoding='utf-8') as f:
    json.dump(bundle, f, ensure_ascii=False, indent=2)
with open(f'{out}/panels_4326.geojson','w',encoding='utf-8') as f:
    json.dump({'type':'FeatureCollection','features':panels_features}, f, ensure_ascii=False)
print(f'저장: {out}/bundle.json, panels_4326.geojson ({len(panels_features)} cells)')

## 다음 단계 (개발자 핸드오프)

- `bundle.json` 의 4326 polygon + 메타 + 음영 통계를 풀스택에 그대로 넘김.
- `panels_4326.geojson` 은 지도 위 셀별 음영 히트맵 표시용.
- 안정성 sweep(반복 호출/좌표 변형) 은 별도 노트북으로.